# 02 — Bake-off: Modeling + Evaluation (CRISP-DM Phases 4–5)

Trains every candidate under one fixed protocol, then evaluates + scores them.
**Run this on Colab (T4)** so all candidates use the same batch size = fair comparison.

Candidates: `cnn_scratch` (from scratch) vs `mobilenetv2`, `efficientnetb0`, `resnet50`.

In [ ]:
# Colab setup (skip locally)
# !git clone <your-repo-url>
# %cd groopy
# !pip install -r requirements.txt
# from google.colab import files; files.upload()  # upload kaggle.json
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !python data/download_asl_alphabet.py
import sys, os
sys.path.insert(0, os.path.abspath('..'))

In [ ]:
# Confirm the GPU is visible
import tensorflow as tf
print('TF', tf.__version__, '| GPUs:', tf.config.list_physical_devices('GPU'))

In [ ]:
# Train the whole bake-off (same protocol for every model). ~ set epochs as time allows.
from recognition.src.train import train_one
from recognition.src.data import make_datasets
from recognition.src import models as zoo

train_ds, val_ds, test_ds, class_names = make_datasets()
records = [train_one(name, epochs=20, train_ds=train_ds, val_ds=val_ds) for name in zoo.REGISTRY]
records

In [ ]:
# Evaluate all trained models + build the scorecard and pick a winner
from recognition.src.evaluate import evaluate_model
from recognition.src import scorecard as sc
from recognition.src.config import MODELS_DIR
from pathlib import Path

rows = [evaluate_model(p.stem, p, test_ds, class_names) for p in sorted(Path(MODELS_DIR).glob('*.keras'))]
ranked = sc.score(rows)
import pandas as pd; pd.DataFrame(ranked)[['model','accuracy','macro_f1','latency','size','total']]

## After training
1. Open `recognition/results/gradcam_*.png` (run `xai_gradcam.py`) and update each model's
   **robustness** score, plus **stability** from the live desktop test.
2. Re-run the scorecard cell to finalise the winner.
3. Export the winner: `python -m recognition.src.export --model recognition/models/<winner>.keras --target all`.